In [0]:
from collections import defaultdict
from pathlib import Path
from typing import Dict, List
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as f
import pyarrow.parquet as pq
from torch.utils.data import IterableDataset, DataLoader
import pickle
import os
import json

In [0]:
# ============================================================
# Parquet Streaming Dataset
# ============================================================
class RankingDataset(IterableDataset):
    def __init__(self, parquet_dir, 
                 user_id_col, post_id_col,
                 user_cat_cols, user_num_cols,
                 post_cat_cols, post_num_cols,
                 user_emb_col, post_emb_col,
                 batch_size=1024):
        super().__init__()
        self.user_id_col, self.post_id_col = user_id_col, post_id_col
        self.user_cat_cols, self.user_num_cols = user_cat_cols, user_num_cols
        self.post_cat_cols, self.post_num_cols = post_cat_cols, post_num_cols
        self.user_emb_col, self.post_emb_col = user_emb_col, post_emb_col
        self.batch_size = batch_size

        self.parquet_files = list(Path(parquet_dir).glob("*.parquet"))

    def __iter__(self):
        for parquet_file in self.parquet_files:
            pf = pq.ParquetFile(parquet_file)
            for batch in pf.iter_batches(batch_size=self.batch_size):
                pdf = batch.to_pandas()
                user_id = torch.tensor(pdf[self.user_id_col].values, dtype=torch.long)
                post_id = torch.tensor(pdf[self.post_id_col].values, dtype=torch.long)

                user_cat = torch.stack([torch.tensor(pdf[c].values, dtype=torch.long) for c in self.user_cat_cols], dim=1)
                user_num = torch.stack([torch.tensor(pdf[c].values, dtype=torch.float32) for c in self.user_num_cols], dim=1)

                post_cat = torch.stack([torch.tensor(pdf[c].values, dtype=torch.long) for c in self.post_cat_cols], dim=1)
                post_num = torch.stack([torch.tensor(pdf[c].values, dtype=torch.float32) for c in self.post_num_cols], dim=1)

                # embeddings stored as arrays in parquet → each row is numpy array
                lastn_embs = torch.tensor(np.stack(pdf[self.user_emb_col].values), dtype=torch.float32)
                post_emb = torch.tensor(np.stack(pdf[self.post_emb_col].values), dtype=torch.float32)                

                labels = {
                    "click": torch.tensor(pdf["label_click"].values, dtype=torch.float32),
                    "like": torch.tensor(pdf["label_like"].values, dtype=torch.float32),
                    "comment": torch.tensor(pdf["label_comment"].values, dtype=torch.float32),
                    "share": torch.tensor(pdf["label_share"].values, dtype=torch.float32),
                    "dwell": torch.tensor(pdf["label_dwell_time"].fillna(0).values, dtype=torch.float32),
                }
                yield {"user_id": user_id,
                        "user_cat": user_cat,
                        "user_num": user_num,
                        "lastn_embs": lastn_embs,
                        "post_id": post_id,
                        "post_cat": post_cat,
                        "post_num": post_num,
                        "post_emb": post_emb, 
                        "labels": labels}            

In [0]:
# ============================================================
# Model: shared trunk + heads
# ============================================================
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used to process concatenated feature vectors.
    Consists of several linear layers with ReLU activations and dropout for regularization.
    """
    def __init__(self, in_dim, hidden_dims=(256,128), dropout=0.2):
        super().__init__()
        layers = []  # List to hold layers
        prev = in_dim  # Track input dimension for each layer
        for h in hidden_dims:
            # Add a Linear layer, followed by ReLU and Dropout
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h  # Update input dimension for next layer
        self.net = nn.Sequential(*layers)  # Combine layers into a sequential model

    def forward(self, x):
        # Forward pass through the MLP
        return self.net(x)

def _last_linear_out_features(seq):
    """
    Utility to get output dimension of last Linear layer in a Sequential.
    """
    for l in reversed(seq):
        if isinstance(l, nn.Linear): return l.out_features
    raise ValueError("No Linear found")

def calibrate_probability(p_pred, alpha: float):
    """
    Calibrate probability after training based on negative downsampling.
    Args:
        p_pred: raw sigmoid probability (tensor)
        alpha: global downsampling ratio (0 < alpha < 1)
    Returns:
        Calibrated probability tensor
    """
    return (alpha * p_pred) / ((1 - p_pred) + alpha * p_pred)

class MultiTaskRanker(nn.Module):
    def __init__(self, uid_vocab, uid_dim, user_cat_dims, user_num_dim, lastn_emb_dim,
                 pid_vocab, pid_dim, post_cat_dims, post_num_dim, post_emb_dim, 
                 hidden_dims=(256,128), dropout=0.2, alpha=1):
        super().__init__()
        # Embedding for user ID (with padding index 0)
        self.user_id_emb = nn.Embedding(uid_vocab+1, uid_dim, padding_idx=0)
        # Embeddings for each categorical user feature
        self.user_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in user_cat_dims])

        # Embedding for post ID (with padding index 0)
        self.post_id_emb = nn.Embedding(pid_vocab+1, pid_dim, padding_idx=0)
        # Embeddings for each categorical post feature
        self.post_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in post_cat_dims])

        u_dim = uid_dim + sum(d for _, d in user_cat_dims) + user_num_dim + lastn_emb_dim
        p_dim = pid_dim + sum(d for _, d in post_cat_dims) + post_num_dim + post_emb_dim

        # MLP to combine all features into a single vector
        self.mlp = MLP(u_dim+p_dim, hidden_dims, dropout)

        last_size = _last_linear_out_features(self.mlp.net)
        self.head_click = nn.Linear(last_size, 1)
        self.head_like = nn.Linear(last_size, 1)
        self.head_comment = nn.Linear(last_size, 1)
        self.head_share = nn.Linear(last_size, 1)
        # dwell head predicts z, with exp(z) ≈ dwell_time
        self.head_dwell = nn.Linear(last_size, 1)

        self.alpha = alpha  # global downsampling ratio

    def forward(self, user_id, user_cat, user_num, lastn_embs, post_id, post_cat, post_num, post_emb, output_type='logit'):
        # Embed user IDs
        u_id = self.user_id_emb(user_id)
        # Embed and concatenate all categorical features (if any)
        if self.user_cat_embs:
            u_cat = torch.cat([emb(user_cat[:, i]) for i, emb in enumerate(self.user_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            u_cat = torch.zeros(u_id.size(0), 0, device=u_id.device)

        # Embed post ID
        p_id = self.post_id_emb(post_id)
        # Embed and concatenate all categorical features (if any)
        if self.post_cat_embs:
            p_cat = torch.cat([emb(post_cat[:, i]) for i, emb in enumerate(self.post_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            p_cat = torch.zeros(p_id.size(0), 0, device=p_id.device)

        # Concatenate all features and pass through MLP
        h = self.mlp(torch.cat([u_id, u_cat, user_num, lastn_embs, p_id, p_cat, post_num, post_emb], dim=-1))

        if output_type == 'logit':
            # Raw logit outputs
            click_pred = self.head_click(h).squeeze(-1)
            like_pred = self.head_like(h).squeeze(-1)
            comment_pred = self.head_comment(h).squeeze(-1)
            share_pred = self.head_share(h).squeeze(-1)
            # Dwell time: transformation base on [P Covington, et.al. Deep Neural Networks for YouTube Recommendations. In RecSys, 2016]
            dwell_pred = self.head_dwell(h).squeeze(-1)
        elif output_type == 'prob':
            # Raw probablity outputs
            click_pred = torch.sigmoid(self.head_click(h).squeeze(-1))
            like_pred = torch.sigmoid(self.head_like(h).squeeze(-1))
            comment_pred = torch.sigmoid(self.head_comment(h).squeeze(-1))
            share_pred = torch.sigmoid(self.head_share(h).squeeze(-1))
            dwell_pred = torch.sigmoid(self.head_dwell(h).squeeze(-1))
        elif output_type == 'calibrated_prob':
            # Apply calibration to probablities when downsampling applied to imbalanced class
            # Xinran He et al. Practical lessons from predicting clicks on ads at Facebook. In the 8th International Workshop on Data Mining for Online Advertising.
            click_pred = calibrate_probability(torch.sigmoid(self.head_click(h).squeeze(-1)), self.alpha)
            like_pred = calibrate_probability(torch.sigmoid(self.head_like(h).squeeze(-1)), self.alpha)
            comment_pred = calibrate_probability(torch.sigmoid(self.head_comment(h).squeeze(-1)), self.alpha)
            share_pred = calibrate_probability(torch.sigmoid(self.head_share(h).squeeze(-1)), self.alpha)
            dwell_pred = torch.sigmoid(self.head_dwell(h).squeeze(-1))  # no calibration for dwell time
        else:
            ValueError("Output_type must be logit, prob or calibrated_prob")

        return {
            "click": click_pred,
            "like": like_pred,
            "comment": comment_pred,
            "share": share_pred,
            "dwell": dwell_pred,
        }

In [0]:
# -----------------------------
# Training loop
# -----------------------------
class RankerTrainer:
    """
    Enhanced trainer that supports multiple negative sampling strategies.
    Supports different objectives and negative sampling strategies.
    """
    def __init__(self, model, lr=1e-3, device="cpu"):
        self.model = model.to(device)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.device = device

    def _bce_loss_fn(self, output_type='logit', pos_weight=None):
        if output_type=='logit':
            return nn.BCEWithLogitsLoss(pos_weight = pos_weight) 
        else:
            return nn.BCELoss()
        
    def _dwell_loss_fn(self, logits: torch.Tensor, true_dwell: torch.Tensor):
        """
        Custom loss: if t = true dwell time,
        y = t/(1+t), p = sigmoid(z), loss = -(t*log(p) + log(1-p)).
        """
        p = torch.sigmoid(logits)
        t = true_dwell
        loss = -(t * torch.log(p + 1e-9) + torch.log(1 - p + 1e-9))
        return loss.mean()

    def _batch_to_device(self, batch):
        # Moves batch tensors to the target device
        td = lambda x: x.to(self.device)
        feats = (td(batch['user_id']), td(batch['user_cat']), td(batch['user_num']), td(batch['lastn_embs']),
                td(batch['post_id']), td(batch['post_cat']), td(batch['post_num']), td(batch['post_emb']))
        
        labels = batch['labels']
        labs = (td(labels['click']), td(labels['like']), td(labels['comment']), td(labels['share']), td(labels['dwell']))

        return feats, labs    

    def train_one_epoch(self, loader, output_type='logit', task_weights={k:1 for k in ('click','like','comment','share','dwell')}, pos_weights=None):
        self.model.train()
        total_loss = 0.0
        
        n_batches = 0
        for batch in loader:
            features, labels = self._batch_to_device(batch)
            preds = self.model(*features, output_type=output_type)
            
            lab_click, lab_like, lab_comment, lab_share, lab_dwell = labels

            if not pos_weights:
                # When no predefined positive-negative class ratio presents, use in-batch sample to estimate the class imbalance ratio
                pos_weights={
                    'click':torch.sum(lab_click==0)/torch.sum(lab_click==1),
                    'like':torch.sum(lab_like==0)/torch.sum(lab_like==1),
                    'comment':torch.sum(lab_comment==0)/torch.sum(lab_comment==1),
                    'share':torch.sum(lab_share==0)/torch.sum(lab_share==1)
                }

            loss_fn_click = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('click', 1)]))
            loss_fn_like = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('like', 1)]))
            loss_fn_comment = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('comment', 1)]))
            loss_fn_share = self._bce_loss_fn(pos_weight=torch.tensor([pos_weights.get('share', 1)]))
            loss_fn_dwell = self._bce_loss_fn(pos_weight=None)

            self.opt.zero_grad()
            loss = 0.0
            loss += task_weights.get('click',1.0) * loss_fn_click(preds['click'], lab_click)
            loss += task_weights.get('like',1.0) * loss_fn_click(preds['like'], lab_like)
            loss += task_weights.get('comment',1.0) * loss_fn_comment(preds['comment'], lab_comment)
            loss += task_weights.get('share',1.0) * loss_fn_comment(preds['share'], lab_share)
            # loss += task_weights.get('dwell', 1.0) * self._dwell_loss_fn(preds['dwell'], lab_dwell)
            # transform lab_dwell: y = t/(1+t)
            loss += task_weights.get('dwell', 1.0) * loss_fn_dwell(preds['dwell'], lab_dwell/(1 + lab_dwell))

            loss.backward()
            self.opt.step()
            total_loss += loss.item()
            n_batches += 1

        return total_loss / n_batches

#### Run training pipeline

In [0]:
# Get pipeline parameters
model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
user_id_col = 'user_id'
post_id_col = 'post_id'
user_cat_cols = ["ip_province", "vehicle_series", "platform"]
user_num_cols = ["months_since_registration", "months_since_consent", "identity", "is_koc", "has_profile_photo",
                "community_visits", "posts_published", "posts_viewed", "posts_liked", 
                "posts_commented", "posts_shared", "users_followed", "tribes_joined"]
post_cat_cols = ["post_type"]
post_num_cols = ["days_since_published", "is_ugc", "is_sink", "has_pic", "has_video",
                 "views", "likes", "comments", "shares", "dwell_time"]
user_emb_col = 'lastn_embs'
post_emb_col = 'post_content_emb'
label_cols = ['label_click', 'label_like', 'label_comment', 'label_share', 'label_dwell_time']

cat_cols = user_cat_cols + post_cat_cols
num_cols = user_num_cols + post_num_cols
emb_cols = [user_emb_col] + [post_emb_col]

user_feature_cols = user_cat_cols + user_num_cols + [user_emb_col]
post_feature_cols = post_cat_cols + post_num_cols + [post_emb_col]


ranking_parquet_dir = model_config['RANKING_TRAIN_PARQUET_PATH']
ranking_metadata_dir = model_config['RANKING_METADATA_PATH']
retrieval_metadata_dir = model_config['TWO_TOWER_METADATA_PATH']

neg_ratio = model_config['neg_sample_ratio']
batch_size = model_config['batch_size']

In [0]:
metadata = pickle.load(open(Path(retrieval_metadata_dir) / 'metadata_stats.pkl', 'rb'))
class_ratios = pickle.load(open(Path(ranking_metadata_dir) / 'class_ratios.pkl', 'rb'))
print("Data stats:")
print(f"Number of users: {metadata['n_users']}")
print(f"Number of posts: {metadata['n_posts']}")
print(f"user categorical dims: {metadata['user_cat_dims']}")
print(f"post categorical dims: {metadata['post_cat_dims']}")
print(f"user numerical dim: {metadata['user_num_dim']}")
print(f"post numerical dim: {metadata['post_num_dim']}")
class_ratios_display = {k.replace('label_', ''): v for k, v in class_ratios.items()}
print(f"Class ratios: {class_ratios_display}")

In [0]:
# Create datasets
train_dataset = RankingDataset(
    parquet_dir=ranking_parquet_dir,
    user_id_col=user_id_col+'_idx', 
    post_id_col=post_id_col+'_idx',
    user_cat_cols=[c+'_idx' for c in user_cat_cols],
    user_num_cols=[c+'_norm' for c in user_num_cols],
    post_cat_cols=[c+'_idx' for c in post_cat_cols],
    post_num_cols=[c+'_norm' for c in post_num_cols],
    user_emb_col=user_emb_col,
    post_emb_col=post_emb_col,
    batch_size=batch_size
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=None, num_workers=0)

print(f"Train dataset: {len(train_dataset.parquet_files)} parquet files")

In [0]:
model_args = model_config['model_args']
user_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['user_cat_dims']]
post_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['post_cat_dims']]

model = MultiTaskRanker(uid_vocab=metadata['n_users'], 
                        uid_dim=model_args['id_emb_dim'],
                        user_cat_dims=user_cat_dims,
                        user_num_dim=metadata['user_num_dim'],
                        lastn_emb_dim=model_args['emb_dim'],
                        pid_vocab=metadata['n_posts'], 
                        pid_dim=model_args['id_emb_dim'],
                        post_cat_dims=post_cat_dims,
                        post_num_dim=metadata['post_num_dim'],
                        post_emb_dim=model_args['emb_dim'], 
                        hidden_dims=model_args['hidden_dims'], 
                        dropout=model_args['dropout'], 
                        alpha=neg_ratio)

In [0]:
def train_streaming(trainer, train_loader, output_type='logit', task_weights={k:1 for k in ('click','like','comment','share','dwell')}, pos_weights=None, epochs=5):
    """
    Training loop specifically for streaming datasets.
    """
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(1, epochs+1):
        # Training
        loss = trainer.train_one_epoch(train_loader, output_type, task_weights, pos_weights)
        
        print(f"Epoch {epoch}:")
        print(f"  Training Loss: {loss:.4f}")

    return trainer.model  

trainer = RankerTrainer(model, lr=model_config['trainer_args']['lr'], device="cpu")
output_type = model_config['output_type']
task_weights = model_config['task_weights']
pos_weights = {k: 1/v for k,v in class_ratios.items()}
epochs = model_config['epochs']

model = train_streaming(
        trainer,
        train_loader,
        output_type,
        task_weights,
        pos_weights,
        epochs=epochs
    )

In [0]:
model_path = Path(ranking_metadata_dir).joinpath('ranking_model')
torch.save(model.state_dict(), model_path)